# BARAM 2026 — MLP random search 튜닝 + 최종 v4

v3의 MLP(고정 설정 128×2, dropout 0.1, lr 1e-3)를 **random search 20개 설정**으로 튜닝.

- 탐색 공간: hidden {64,128,256} × depth {2,3} × dropout {0,0.1,0.2,0.3} × lr log[3e-4,3e-3] × wd {1e-5,1e-4,1e-3} × batch {256,512,1024} × emb {2,4,8}
- 평가: expanding CV 2폴드(→2023, →2024) 공식 총점. **GBM 폴드 예측은 1회 계산 후 캐시**(변하지 않음).
- 각 설정마다 블렌드 가중치 w ∈ {0.25,0.4,0.5,0.6}도 무료로 스캔.
- 채택: **두 해 모두 GBM 단독보다 우위**인 것 중 평균 총점 최대 (v3와 동일 규율).
- 최종: 튜닝 설정으로 전체 재학습 + FICR 후처리 → `submission_v4.csv`.

## 0. 설정 & GBM 폴드 예측 캐시

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS","1")
import warnings; warnings.filterwarnings("ignore")
import json, numpy as np, pandas as pd
import torch, torch.nn as nn
torch.set_num_threads(1)
import lightgbm as lgb
from sklearn.ensemble import HistGradientBoostingRegressor as HGB
import wind_lib as W
from official_metric import group_scores, metric

GROUPS=(1,2,3); FR={}; TGT={}; FR_TE={}
for g in GROUPS:
    df,tgt=W.load_train(g); TGT[g]=tgt
    FR[g]=W.add_spatial(W.build(df,g),"train")
    FR_TE[g]=W.add_spatial(W.build(W.load_test(g),g),"test")
BASE_ALL=[c for c in W.feature_cols(FR[1]) if c not in W.SPATIAL_COLS]+["pc_pred_cf"]
V2=W.lean_features(BASE_ALL)+W.SPATIAL_COLS

def lgbm(): return lgb.LGBMRegressor(objective="mae",n_estimators=600,learning_rate=0.03,num_leaves=63,
    min_child_samples=60,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,reg_lambda=1.0,
    random_state=42,n_jobs=1,verbose=-1)
def histgbm(): return HGB(loss="absolute_error",max_iter=600,learning_rate=0.03,max_leaf_nodes=63,
    l2_regularization=1.0,random_state=42)
def gbm_ens(tr,va,cols,tgt):
    lg=lgbm().fit(tr[cols],tr[tgt]); hg=histgbm().fit(tr[cols].to_numpy(),tr[tgt].to_numpy())
    return 0.5*(lg.predict(va[cols])+hg.predict(va[cols].to_numpy()))

YEAR_FOLDS=[([2022],2023),([2022,2023],2024)]
CACHE={}   # (vy) -> dict(g -> dict(tr2,va2,gbm))
for tys,vy in YEAR_FOLDS:
    ent={}
    for g in GROUPS:
        tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]; yr=fr.kst_dtm.dt.year
        tr=fr[yr.isin(tys)]; va=fr[yr==vy]
        if len(tr)==0 or len(va)==0: continue
        iso=W.fit_powercurve(tr,tgt,cap)
        tr2=W.with_pc(tr,iso); va2=W.with_pc(va,iso)
        ent[g]=dict(tr2=tr2,va2=va2,gbm=np.clip(gbm_ens(tr2,va2,V2,tgt),0,cap))
    CACHE[vy]=ent
    print(f"cached fold →{vy}: groups {list(ent)}")

def fold_total(preds, vy):
    """preds: {g: pred} → 공식 총점(해당 폴드 그룹 평균)."""
    nm=[]; fi=[]
    for g,p in preds.items():
        tgt=TGT[g]; cap=W.CAP[g]; yt=CACHE[vy][g]["va2"][tgt].to_numpy()
        a,b=group_scores(yt,np.clip(p,0,cap),cap); nm.append(a); fi.append(b)
    return 0.5*(1-np.mean(nm))+0.5*np.mean(fi)

GBM_TOTAL={vy: fold_total({g:CACHE[vy][g]["gbm"] for g in CACHE[vy]}, vy) for vy in (2023,2024)}
print("GBM 단독 총점:", {k:round(v,4) for k,v in GBM_TOTAL.items()})

cached fold →2023: groups [1, 2]


cached fold →2024: groups [1, 2, 3]
GBM 단독 총점: {2023: np.float64(0.5862), 2024: np.float64(0.6013)}


## 1. MLP (파라미터화) & random search

In [2]:
class MLP(nn.Module):
    def __init__(s,nf,ng=3,h=128,depth=2,drop=0.1,emb=4):
        super().__init__(); s.emb=nn.Embedding(ng,emb)
        layers=[nn.Linear(nf+emb,h),nn.GELU(),nn.Dropout(drop)]
        for _ in range(depth-1):
            layers+=[nn.Linear(h,h),nn.GELU(),nn.Dropout(drop)]
        layers+=[nn.Linear(h,1)]
        s.net=nn.Sequential(*layers)
    def forward(s,x,g): return s.net(torch.cat([x,s.emb(g)],-1)).squeeze(-1)

def train_mlp(pool_tr,feats,cfg,seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    pool_tr=pool_tr.sort_values("kst_dtm")
    mu=pool_tr[feats].mean(); sd=pool_tr[feats].std()+1e-8
    X=((pool_tr[feats]-mu)/sd).to_numpy(np.float32)
    y=pool_tr["cf"].to_numpy(np.float32); gid=pool_tr["gid"].to_numpy(np.int64)
    n=len(X); cut=int(n*0.85)
    Xt,yt,gt=torch.tensor(X),torch.tensor(y),torch.tensor(gid)
    net=MLP(len(feats),h=cfg["h"],depth=cfg["depth"],drop=cfg["drop"],emb=cfg["emb"])
    opt=torch.optim.AdamW(net.parameters(),lr=cfg["lr"],weight_decay=cfg["wd"])
    best=1e9; st=None; pat=0
    for ep in range(120):
        net.train(); perm=np.random.permutation(np.arange(cut))
        for i in range(0,len(perm),cfg["bs"]):
            b=perm[i:i+cfg["bs"]]; opt.zero_grad()
            ((net(Xt[b],gt[b])-yt[b]).abs().mean()).backward(); opt.step()
        net.eval()
        with torch.no_grad(): vl=(net(Xt[cut:],gt[cut:])-yt[cut:]).abs().mean().item()
        if vl<best-1e-5: best=vl; st={k:v.clone() for k,v in net.state_dict().items()}; pat=0
        else: pat+=1
        if pat>=10: break
    net.load_state_dict(st); return net,(mu,sd)

def mlp_predict(net,scaler,fr,feats,g,cap):
    mu,sd=scaler
    X=torch.tensor(((fr[feats]-mu)/sd).to_numpy(np.float32))
    gid=torch.full((len(fr),),g-1,dtype=torch.long)
    net.eval()
    with torch.no_grad(): p=net(X,gid).numpy()
    return np.clip(p,0,1)*cap

def make_pool(vy):
    parts=[]
    for g,e in CACHE[vy].items():
        p=e["tr2"][V2+["kst_dtm"]].copy(); p["cf"]=e["tr2"][TGT[g]]/W.CAP[g]; p["gid"]=g-1; parts.append(p)
    return pd.concat(parts,ignore_index=True)

# random search
rng=np.random.RandomState(123)
def sample_cfg():
    return dict(h=int(rng.choice([64,128,256])), depth=int(rng.choice([2,3])),
                drop=float(rng.choice([0.0,0.1,0.2,0.3])),
                lr=float(10**rng.uniform(np.log10(3e-4),np.log10(3e-3))),
                wd=float(rng.choice([1e-5,1e-4,1e-3])),
                bs=int(rng.choice([256,512,1024])), emb=int(rng.choice([2,4,8])))

N_TRIALS=20; WEIGHTS=[0.25,0.4,0.5,0.6]
results=[]
for t in range(N_TRIALS):
    cfg=sample_cfg()
    mlp_pred={}   # vy -> {g: pred}
    ok=True
    for vy in (2023,2024):
        pool=make_pool(vy)
        net,scaler=train_mlp(pool,V2,cfg,seed=t)
        mlp_pred[vy]={g:mlp_predict(net,scaler,CACHE[vy][g]["va2"],V2,g,W.CAP[g]) for g in CACHE[vy]}
    row=dict(trial=t,**cfg)
    for w in WEIGHTS:
        tot={}
        for vy in (2023,2024):
            blend={g:(1-w)*CACHE[vy][g]["gbm"]+w*mlp_pred[vy][g] for g in CACHE[vy]}
            tot[vy]=fold_total(blend,vy)
        row[f"w{w}_2023"]=tot[2023]; row[f"w{w}_2024"]=tot[2024]
        row[f"w{w}_mean"]=0.5*(tot[2023]+tot[2024])
        row[f"w{w}_beats"]=(tot[2023]>GBM_TOTAL[2023]) and (tot[2024]>GBM_TOTAL[2024])
    results.append(row)
    bestw=max(WEIGHTS,key=lambda w:row[f"w{w}_mean"])
    print(f"trial {t:2d} h={cfg['h']} d={cfg['depth']} drop={cfg['drop']} lr={cfg['lr']:.4f} "
          f"wd={cfg['wd']:.0e} bs={cfg['bs']} emb={cfg['emb']} | best w={bestw} "
          f"mean={row[f'w{bestw}_mean']:.4f} beats_both={row[f'w{bestw}_beats']}")
rs=pd.DataFrame(results)

trial  0 h=256 d=3 drop=0.2 lr=0.0008 wd=1e-03 bs=1024 emb=4 | best w=0.5 mean=0.5980 beats_both=True


trial  1 h=256 d=3 drop=0.1 lr=0.0029 wd=1e-05 bs=512 emb=8 | best w=0.25 mean=0.5959 beats_both=True


trial  2 h=128 d=2 drop=0.2 lr=0.0013 wd=1e-04 bs=1024 emb=4 | best w=0.4 mean=0.5989 beats_both=True


trial  3 h=64 d=2 drop=0.0 lr=0.0016 wd=1e-03 bs=256 emb=8 | best w=0.4 mean=0.5953 beats_both=True


trial  4 h=64 d=3 drop=0.0 lr=0.0006 wd=1e-04 bs=256 emb=2 | best w=0.5 mean=0.5999 beats_both=True


...

trial 15 h=256 d=3 drop=0.0 lr=0.0016 wd=1e-04 bs=256 emb=4 | best w=0.6 mean=0.6063 beats_both=True


trial 16 h=64 d=3 drop=0.2 lr=0.0006 wd=1e-05 bs=512 emb=4 | best w=0.6 mean=0.5978 beats_both=True


trial 17 h=128 d=2 drop=0.3 lr=0.0011 wd=1e-05 bs=1024 emb=4 | best w=0.25 mean=0.5950 beats_both=False


trial 18 h=256 d=2 drop=0.1 lr=0.0003 wd=1e-05 bs=256 emb=8 | best w=0.5 mean=0.5991 beats_both=True


trial 19 h=64 d=2 drop=0.1 lr=0.0013 wd=1e-05 bs=256 emb=2 | best w=0.4 mean=0.5979 beats_both=True


## 2. 최적 설정 선택

In [3]:
# 두 해 모두 GBM을 이기는 (trial, w) 중 평균 총점 최대
cands=[]
for _,r in rs.iterrows():
    for w in WEIGHTS:
        if r[f"w{w}_beats"]:
            cands.append((r["trial"],w,r[f"w{w}_mean"],r[f"w{w}_2023"],r[f"w{w}_2024"]))
if cands:
    best_trial,best_w,best_mean,b23,b24=max(cands,key=lambda x:x[2])
else:  # 안전장치: v3 기본설정 유지
    best_trial,best_w=None,0.5
print(f"채택: trial={best_trial}, w={best_w}, mean={best_mean:.4f} (2023={b23:.4f}, 2024={b24:.4f})")
print(f"참고 GBM 단독: 2023={GBM_TOTAL[2023]:.4f}, 2024={GBM_TOTAL[2024]:.4f}")
print(f"참고 v3(B50 고정설정): 2023=0.5915, 2024=0.6028")
BEST_CFG={k:rs.loc[rs.trial==best_trial,k].values[0] for k in ["h","depth","drop","lr","wd","bs","emb"]}
BEST_CFG={k:(int(v) if k in ("h","depth","bs","emb") else float(v)) for k,v in BEST_CFG.items()}
BEST_SEED=int(best_trial)
print("BEST_CFG:",BEST_CFG,"w=",best_w)
top5=sorted(cands,key=lambda x:-x[2])[:5]
print("\nTop5 (trial,w,mean):",[(int(a),b,round(c,4)) for a,b,c,_,_ in top5])

채택: trial=15, w=0.6, mean=0.6063 (2023=0.6004, 2024=0.6123)
참고 GBM 단독: 2023=0.5862, 2024=0.6013
참고 v3(B50 고정설정): 2023=0.5915, 2024=0.6028
BEST_CFG: {'h': 256, 'depth': 3, 'drop': 0.0, 'lr': 0.0015868563457489381, 'wd': 0.0001, 'bs': 256, 'emb': 4} w= 0.6

Top5 (trial,w,mean): [(15, 0.6, 0.6063), (15, 0.5, 0.6054), (15, 0.4, 0.6046), (11, 0.5, 0.6009), (11, 0.4, 0.6008)]


## 3. 최종: 튜닝 MLP로 v4 파이프라인 (holdout 후처리 → 전체 재학습 → 제출)

In [4]:
BLEND=best_w
def blend_predict(tr_frames):
    pool=[]
    for g,(tr2,_) in tr_frames.items():
        p=tr2[V2+["kst_dtm"]].copy(); p["cf"]=tr2[TGT[g]]/W.CAP[g]; p["gid"]=g-1; pool.append(p)
    net,scaler=train_mlp(pd.concat(pool,ignore_index=True),V2,BEST_CFG,seed=BEST_SEED)
    out={}
    for g,(tr2,va2) in tr_frames.items():
        cap=W.CAP[g]
        gp=np.clip(gbm_ens(tr2,va2,V2,TGT[g]),0,cap)
        mp=mlp_predict(net,scaler,va2,V2,g,cap)
        out[g]=np.clip((1-BLEND)*gp+BLEND*mp,0,cap)
    return out

# holdout 예측 + OOF
tr_frames={}
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]; fr=FR[g]
    m_tr=fr.kst_dtm<W.VALID_START; m_va=fr.kst_dtm>=W.VALID_START
    iso=W.fit_powercurve(fr[m_tr],tgt,cap)
    tr_frames[g]=(W.with_pc(fr[m_tr],iso), W.with_pc(fr[m_va],iso))
val_pred=blend_predict(tr_frames)

OOF={}
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]; tr2=tr_frames[g][0]
    oof=np.full(len(tr2),np.nan); years=sorted(tr2.kst_dtm.dt.year.unique())
    if len(years)>=2:
        for ty in years:
            m_in=tr2.kst_dtm.dt.year!=ty; m_out=(tr2.kst_dtm.dt.year==ty).to_numpy()
            oof[m_out]=blend_predict({g:(tr2[m_in],tr2[tr2.kst_dtm.dt.year==ty])})[g]
    else:
        n=len(tr2); cut=int(n*0.7)
        oof[cut:]=blend_predict({g:(tr2.iloc[:cut],tr2.iloc[cut:])})[g]
    OOF[g]=oof

def debias_fit(tr,tgt,oof):
    d=tr.assign(oof=oof); d=d[np.isfinite(d["oof"])].copy(); d["resid"]=d[tgt]-d["oof"]
    d["lb"]=pd.cut(d["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
    d["wq"]=pd.qcut(d["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
    return d.groupby(["lb","wq"])["resid"].mean(), d["resid"].mean()
def debias_apply(va,pred,tbl,glob):
    v=va.copy()
    v["lb"]=pd.cut(v["lead_h"],bins=[15,21,27,33,40],labels=False,include_lowest=True)
    v["wq"]=pd.qcut(v["gfs_wind_speed_100m_mean"],5,labels=False,duplicates="drop")
    return pred+np.array([tbl.get(k,glob) for k in zip(v["lb"],v["wq"])])
def nudge_fit(tr,tgt,oof,cap):
    d=tr.assign(oof=oof); d=d[np.isfinite(d["oof"])]
    yt=d[tgt].to_numpy(); yp=d["oof"].to_numpy(); best=(1.0,0.0); bf=-1
    for s in np.linspace(0.90,1.15,26):
        for sh in np.linspace(-0.06,0.06,25)*cap:
            f=group_scores(yt,np.clip(yp*s+sh,0,cap),cap)[1]
            if f>bf: bf=f; best=(s,sh)
    return best

STORE={}; choice={}
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]; tr2,va2=tr_frames[g]; bp=val_pred[g]
    tbl,glob=debias_fit(tr2,tgt,OOF[g]); s,sh=nudge_fit(tr2,tgt,OOF[g],cap)
    p1=np.clip(debias_apply(va2,bp,tbl,glob),0,cap)
    cand={"P0_none":bp,"P1_debias":p1,"P3_nudge":np.clip(bp*s+sh,0,cap),
          "P5_deb_nudge":np.clip(p1*s+sh,0,cap)}
    STORE[g]=dict(tbl=tbl,glob=glob,nudge=(s,sh))
    rows=[]
    for name,p in cand.items():
        nm,fi=group_scores(va2[tgt].to_numpy(),p,cap); rows.append(dict(post=name,nmae=nm,ficr=fi,contrib=fi-nm))
    t=pd.DataFrame(rows).set_index("post"); choice[g]=t["contrib"].idxmax()
    print(f"group{g}: {choice[g]}")

def apply_post(g,frame,pred):
    cap=W.CAP[g]; st=STORE[g]; ch=choice[g]
    if ch=="P0_none": return pred
    if ch=="P1_debias": return np.clip(debias_apply(frame,pred,st["tbl"],st["glob"]),0,cap)
    if ch=="P3_nudge": return np.clip(pred*st["nudge"][0]+st["nudge"][1],0,cap)
    q=np.clip(debias_apply(frame,pred,st["tbl"],st["glob"]),0,cap)
    return np.clip(q*st["nudge"][0]+st["nudge"][1],0,cap)

ans=pd.DataFrame({f"kpx_group_{g}":tr_frames[g][1][TGT[g]].to_numpy() for g in GROUPS})
p0=pd.DataFrame({f"kpx_group_{g}":val_pred[g] for g in GROUPS})
p1df=pd.DataFrame({f"kpx_group_{g}":apply_post(g,tr_frames[g][1],val_pred[g]) for g in GROUPS})
t0=metric(ans,p0); t1=metric(ans,p1df)
print(f"\n2024 총점: 튜닝B{BLEND} {t0[0]:.4f} → +후처리 {t1[0]:.4f} (1-NMAE {t1[1]:.4f}, FICR {t1[2]:.4f})")
print(f"cf. v3 = 0.6383")

group1: P5_deb_nudge
group2: P5_deb_nudge
group3: P3_nudge

2024 총점: 튜닝B0.6 0.6123 → +후처리 0.6463 (1-NMAE 0.8701, FICR 0.4225)
cf. v3 = 0.6383


In [5]:
# 전체 재학습 → 제출
full_frames={}
for g in GROUPS:
    tgt=TGT[g]; cap=W.CAP[g]
    iso=W.fit_powercurve(FR[g],tgt,cap)
    full_frames[g]=(W.with_pc(FR[g],iso), W.with_pc(FR_TE[g],iso))
test_pred=blend_predict(full_frames)

out=W.load_test(1)[["forecast_id","kst_dtm"]].rename(columns={"kst_dtm":"forecast_kst_dtm"})
for g in GROUPS:
    out[f"kpx_group_{g}"]=apply_post(g, full_frames[g][1], test_pred[g])
assert out.shape[0]==8760
for g in GROUPS:
    c=out[f"kpx_group_{g}"]; assert (c>=0).all() and (c<=W.CAP[g]).all() and c.notna().all()
out.to_csv("submission/ver_4/submission.csv",index=False); print("saved submission/ver_4/submission.csv",out.shape)

summary=dict(pipeline=f"V2 65 feats · GBM⊕MLP(tuned, w={BLEND}) · FICR postproc",
    best_cfg=BEST_CFG, best_seed=BEST_SEED, blend_w=float(BLEND), chosen_post=choice,
    holdout_total=round(float(t1[0]),4),
    one_minus_nmae=round(float(t1[1]),4), ficr=round(float(t1[2]),4),
    v3_reference=0.6383)
json.dump(summary,open("submission/ver_4/result.json","w"),ensure_ascii=False,indent=2)
print(json.dumps(summary,ensure_ascii=False,indent=2))

saved submission_v4.csv (8760, 5)
{
  "pipeline": "V2 65 feats · GBM⊕MLP(tuned, w=0.6) · FICR postproc",
  "best_cfg": {
    "h": 256,
    "depth": 3,
    "drop": 0.0,
    "lr": 0.0015868563457489381,
    "wd": 0.0001,
    "bs": 256,
    "emb": 4
  },
  "best_seed": 15,
  "blend_w": 0.6,
  "chosen_post": {
    "1": "P5_deb_nudge",
    "2": "P5_deb_nudge",
    "3": "P3_nudge"
  },
  "holdout_total": 0.6463,
  "one_minus_nmae": 0.8701,
  "ficr": 0.4225,
  "v3_reference": 0.6383
}


## 결론
- random search로 튜닝된 MLP+블렌드 가중치가 CV 두 해 모두에서 GBM 단독·v3 고정설정을 이기는지 확인.
- v4 총점이 v3(0.6383)보다 높으면 `submission_v4.csv`가 새 제출 후보, 아니면 v3 유지.